In [0]:
#Config

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)   # fixed seed = same data every run

N_DAYS = 60
START_DATE = pd.Timestamp("2026-07-01")
N_CUSTOMERS = 20000

In [0]:
#Creating the layer schemas

for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{schema}")

In [0]:
#Stores and daily context

stores = pd.DataFrame({
    "store_id": [f"DS0{i}" for i in range(1, 7)],
    "store_name": [f"Dark Store {i}" for i in range(1, 7)],
    "zone": ["North", "North", "South", "South", "East", "West"],
    "base_daily_orders": [320, 280, 350, 260, 300, 290],  # how busy each store's area is
})

dates = pd.date_range(START_DATE, periods=N_DAYS, freq="D")
daily_context = pd.DataFrame({
    "date": dates.date,
    "is_weekend": dates.dayofweek >= 5,
    "is_rainy": rng.random(N_DAYS) < 0.20,   # roughly 1 rainy day in 5
})

In [0]:
#SKUs

categories = [
    # name, number of SKUs, is_perishable, typical price (INR)
    ("Fruits & Vegetables", 60, True, 60),
    ("Dairy & Eggs", 35, False, 70),
    ("Bakery", 20, False, 50),
    ("Snacks", 50, False, 40),
    ("Beverages", 40, False, 60),
    ("Staples", 35, False, 150),
    ("Personal Care", 30, False, 180),
    ("Household", 30, False, 120),
]

rows, n = [], 1
for name, count, perishable, typical_price in categories:
    for _ in range(count):
        rows.append({
            "sku_id": f"SKU{n:04d}",
            "category": name,
            "is_perishable": perishable,
            "unit_price": int(max(10, rng.lognormal(np.log(typical_price), 0.4))),
            "shelf_life_days": int(rng.integers(2, 6)) if perishable else 180,
            "popularity": float(rng.pareto(1.5) + 1),   # heavy tail: few SKUs dominate
        })
        n += 1
skus = pd.DataFrame(rows)

In [0]:
#Customers

w = stores["base_daily_orders"] / stores["base_daily_orders"].sum()
signup = START_DATE + pd.to_timedelta(rng.integers(-400, N_DAYS, N_CUSTOMERS), unit="D")

customers = pd.DataFrame({
    "customer_id": [f"C{i:06d}" for i in range(1, N_CUSTOMERS + 1)],
    "signup_date": signup,
    "home_store_id": rng.choice(stores["store_id"], N_CUSTOMERS, p=w),
})
customers["signup_date"] = customers["signup_date"].dt.date

In [0]:
#Save to Bronze

def save_bronze(pdf, name):
    sdf = spark.createDataFrame(pdf)
    sdf.write.mode("overwrite").saveAsTable(f"workspace.bronze.{name}")
    print(f"{name}: {sdf.count()} rows")

save_bronze(stores, "stores")
save_bronze(skus, "skus")
save_bronze(customers, "customers")
save_bronze(daily_context, "daily_context")

In [0]:
%sql
--Check

SELECT category, COUNT(*) AS skus, ROUND(AVG(unit_price)) AS avg_price
FROM workspace.bronze.skus
GROUP BY category
ORDER BY skus DESC;

In [0]:
#Demand settings

HOURS = np.arange(6, 24)   # stores operate 06:00-23:59
HOUR_WEIGHTS = np.array([0.5, 1.5, 2.5, 3, 3, 4, 5, 5, 4, 3.5, 4, 5, 6.5, 8.5, 9.5, 9, 6, 3])
HOUR_P = HOUR_WEIGHTS / HOUR_WEIGHTS.sum()   # lunch peak and a bigger dinner peak
assert len(HOURS) == len(HOUR_WEIGHTS)

WEEKEND_MULT = 1.15   # weekends are busier
RAIN_MULT = 1.10      # rain increases orders slightly

In [0]:
#who orders(customer pools)

cust_ids = customers["customer_id"].values
cust_weight = rng.gamma(0.7, 1.0, N_CUSTOMERS) + 0.05   # heavy tail: a few customers order a lot
cust_signup = pd.to_datetime(customers["signup_date"]).values

# customers grouped by their home store
home_idx = {s: np.where(customers["home_store_id"].values == s)[0] for s in stores["store_id"]}

In [0]:
#Generate orders, day by day and store by store

frames = []
for day, ctx in zip(dates, daily_context.itertuples()):
    for s in stores.itertuples():
        mult = (WEEKEND_MULT if ctx.is_weekend else 1.0) * (RAIN_MULT if ctx.is_rainy else 1.0)
        n = rng.poisson(s.base_daily_orders * mult)

        # only customers of this store who have already signed up can order
        pool = home_idx[s.store_id]
        pool = pool[cust_signup[pool] <= day.to_datetime64()]
        p = cust_weight[pool] / cust_weight[pool].sum()

        chosen = rng.choice(pool, n, p=p)
        hour = rng.choice(HOURS, n, p=HOUR_P)
        seconds = hour * 3600 + rng.integers(0, 3600, n)

        frames.append(pd.DataFrame({
            "store_id": s.store_id,
            "customer_id": cust_ids[chosen],
            "placed_ts": day + pd.to_timedelta(seconds, unit="s"),
        }))

orders_raw = pd.concat(frames, ignore_index=True).sort_values("placed_ts").reset_index(drop=True)
orders_raw.insert(0, "order_id", [f"ORD{i:07d}" for i in range(1, len(orders_raw) + 1)])

In [0]:
#Basket size, promise time, and the generator's private plan

n_items = np.minimum(1 + rng.poisson(3.0, len(orders_raw)), 12)   # about 4 lines per order
orders_raw["promised_minutes"] = np.where(n_items <= 4, 10, 15)   # bigger baskets get a longer promise

# n_items is the generator's private knowledge (not in the raw feed); module 3 uses it
order_plan = pd.DataFrame({"order_id": orders_raw["order_id"], "n_items": n_items})

orders = orders_raw[["order_id", "customer_id", "store_id", "placed_ts", "promised_minutes"]]
assert orders["order_id"].is_unique
print(len(orders), "orders")

In [0]:
#save to bronze

save_bronze(orders, "orders")

In [0]:
%sql
--demand curve check

SELECT HOUR(placed_ts) AS hr, COUNT(*) AS orders
FROM workspace.bronze.orders
GROUP BY 1 ORDER BY 1;

In [0]:
%sql
--rain effect check
SELECT c.is_rainy,
       COUNT(DISTINCT c.date) AS days,
       ROUND(COUNT(*) / COUNT(DISTINCT c.date)) AS avg_orders_per_day
FROM workspace.bronze.orders o
JOIN workspace.bronze.daily_context c ON DATE(o.placed_ts) = c.date
GROUP BY c.is_rainy;

In [0]:
#Availability grid (with the planted stockout patterns)

S, K, D = len(stores), len(skus), N_DAYS
store_pos = {s: i for i, s in enumerate(stores["store_id"])}

avail = np.ones((S, K, D, 24), dtype=bool)   # avail[store, sku, day, hour] = True if in stock
pop_pct = skus["popularity"].rank(pct=True).values   # 1.0 = most popular SKU
is_dairy = (skus["category"] == "Dairy & Eggs").values
is_fv = (skus["category"] == "Fruits & Vegetables").values

for si, store_id in enumerate(stores["store_id"]):
    # every SKU can have an outage on any day; popular SKUs run out more often
    q = np.tile(0.04 + 0.10 * pop_pct[:, None], (1, D))   # chance of an outage that day
    starts = rng.integers(6, 21, (K, D))                  # hour the outage starts
    lens = rng.integers(1, 7, (K, D))                     # outage length in hours

    # planted patterns
    if store_id == "DS05":            # understocks fruit & veg
        q[is_fv] = 0.50
        lens[is_fv] = rng.integers(3, 8, (is_fv.sum(), D))
    elif store_id == "DS02":          # overstocks fruit & veg (shows up as wastage later)
        q[is_fv] = 0.01
    else:
        q[is_fv] = 0.10
    if store_id == "DS04":            # dairy restocking gap at the evening peak
        n_d = is_dairy.sum()
        q[is_dairy] = 0.75
        starts[is_dairy] = rng.integers(17, 21, (n_d, D))
        lens[is_dairy] = rng.integers(3, 7, (n_d, D))

    for ki, di in np.argwhere(rng.random((K, D)) < q):
        avail[si, ki, di, starts[ki, di]: starts[ki, di] + lens[ki, di]] = False

In [0]:
#Check the patterns are in the grid

for si, store_id in enumerate(stores["store_id"]):
    a = avail[si][:, :, 6:24]
    print(store_id, "all:", round(1 - a.mean(), 4),
          "| dairy:", round(1 - a[is_dairy].mean(), 4),
          "| F&V:", round(1 - a[is_fv].mean(), 4))

In [0]:
#Order lines (which SKUs each order contains)

qty_choices = np.array([1, 2, 3])
qty_p = np.array([0.70, 0.22, 0.08])
sku_p = (skus["popularity"] / skus["popularity"].sum()).values
n_items_arr = order_plan["n_items"].values

order_idx = np.repeat(np.arange(len(orders)), n_items_arr)
sku_idx = np.concatenate([rng.choice(K, size=k, replace=False, p=sku_p) for k in n_items_arr])

lines = pd.DataFrame({
    "order_idx": order_idx,
    "sku_idx": sku_idx,
    "qty_ordered": rng.choice(qty_choices, len(order_idx), p=qty_p),
})
oi = lines["order_idx"].values
ki_arr = lines["sku_idx"].values
placed = orders["placed_ts"]

lines["order_id"] = orders["order_id"].values[oi]
lines["sku_id"] = skus["sku_id"].values[ki_arr]
lines["unit_price"] = skus["unit_price"].values[ki_arr]

sp = orders["store_id"].map(store_pos).values[oi]                              # store position
dy = (placed.dt.normalize() - START_DATE).dt.days.values[oi]                   # day index
hr = placed.dt.hour.values[oi]                                                 # hour of day

oos = ~avail[sp, ki_arr, dy, hr]      # True = item was out of stock when ordered
print(len(lines), "order lines;", round(oos.mean() * 100, 2), "% unavailable")

In [0]:
#What happens to missing items (split, substitute, or remove)

NEIGHBOR = {"DS01": "DS02", "DS02": "DS01", "DS03": "DS04", "DS04": "DS03", "DS05": "DS06", "DS06": "DS05"}
neighbor_pos = np.array([store_pos[NEIGHBOR[s]] for s in stores["store_id"]])

P_SPLIT = 0.45        # knob: chance an eligible order is split across two stores
P_SUBSTITUTE = 0.55   # knob: chance a missing item is replaced by a similar one

# 1) can the missing item ship from the neighbouring store instead?
nb_stock = avail[neighbor_pos[sp], ki_arr, dy, hr]
can_move = oos & nb_stock
n_move = np.bincount(oi, weights=can_move.astype(float), minlength=len(orders))
eligible = (n_move > 0) & (n_move < n_items_arr)     # something moves AND something stays
is_split = eligible & (rng.random(len(orders)) < P_SPLIT)
moved = can_move & is_split[oi]

# 2) other missing items: substitute with a similar in-stock SKU, or remove the item
rest_oos = oos & ~moved
substituted = rest_oos & (rng.random(len(lines)) < P_SUBSTITUTE)
substitute_sku = np.full(len(lines), None, dtype=object)
cat = skus["category"].values
sku_ids_arr = skus["sku_id"].values
for i in np.where(substituted)[0]:
    candidates = np.where((cat == cat[ki_arr[i]]) & avail[sp[i], :, dy[i], hr[i]])[0]
    if len(candidates) == 0:
        substituted[i] = False            # nothing suitable to substitute with
    else:
        substitute_sku[i] = sku_ids_arr[rng.choice(candidates)]
removed = rest_oos & ~substituted

# 3) build the tables (kept in memory for now)
status = np.where(moved, "fulfilled_split",
         np.where(substituted, "substituted",
         np.where(removed, "removed", "fulfilled")))

order_items = lines[["order_id", "sku_id", "qty_ordered", "unit_price"]].copy()
order_items["qty_fulfilled"] = np.where(removed, 0, lines["qty_ordered"].values)
order_items["item_status"] = status
order_items["sub_order_id"] = np.where(moved, lines["order_id"] + "-S2", lines["order_id"] + "-S1")
order_items["substitute_sku_id"] = substitute_sku

primary = pd.DataFrame({
    "sub_order_id": orders["order_id"] + "-S1", "order_id": orders["order_id"],
    "store_id": orders["store_id"], "sub_order_seq": 1,
})
second = pd.DataFrame({
    "sub_order_id": orders["order_id"][is_split] + "-S2", "order_id": orders["order_id"][is_split],
    "store_id": [NEIGHBOR[s] for s in orders["store_id"][is_split]], "sub_order_seq": 2,
})
sub_orders = (pd.concat([primary, second], ignore_index=True)
                .sort_values(["order_id", "sub_order_seq"]).reset_index(drop=True))

# generator's private notes for the next module (cancellations depend on these)
order_plan["n_oos"] = np.bincount(oi, weights=oos.astype(float), minlength=len(orders)).astype(int)
order_plan["is_split"] = is_split

In [0]:
#Checks

print("orders:", len(orders), "| order_items:", len(order_items), "| sub_orders:", len(sub_orders))
print("split share of orders:", round(is_split.mean() * 100, 2), "%")
print(order_items["item_status"].value_counts(normalize=True).round(4).to_string())

assert order_items.groupby("order_id").size().eq(n_items_arr).all()
assert sub_orders["sub_order_id"].is_unique
assert set(order_items["sub_order_id"]).issubset(set(sub_orders["sub_order_id"]))
print("all checks passed")